---
title: Глава 5. Запросы к нескольким таблицам
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
# subject: SQL Learning & Tooling
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
      # - Data Analyst
      # - BI Analyst
      # - Business Analyst
      # - Independent Researcher
date: 2026-08-10
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="*UHB5rdx",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


## Что такое соединение

In [2]:
%%sql
desc customer;

9 rows affected.

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [3]:
%%sql
desc address;

9 rows affected.

Field,Type,Null,Key,Default,Extra
address_id,smallint unsigned,NO,PRI,None,auto_increment
address,varchar(50),NO,,None,
address2,varchar(50),YES,,None,
district,varchar(20),NO,,None,
city_id,smallint unsigned,NO,MUL,None,
postal_code,varchar(10),YES,,None,
phone,varchar(20),NO,,None,
location,geometry,NO,MUL,None,
last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


### Декартово произведение

In [2]:
import pandas as pd
print(pd.__version__)

3.0.5


In [3]:
%config SqlMagic.autopandas = True

In [6]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c JOIN address a;

361197 rows affected.

,first_name,last_name,address
0,AUSTIN,CINTRON,47 MySakila Drive
1,WADE,DELVALLE,47 MySakila Drive
2,FREDDIE,DUGGAN,47 MySakila Drive
3,ENRIQUE,FORSYTHE,47 MySakila Drive
4,TERRENCE,GUNDERSON,47 MySakila Drive
...,...,...,...
361192,ELIZABETH,BROWN,1325 Fukuyama Street
361193,BARBARA,JONES,1325 Fukuyama Street
361194,LINDA,WILLIAMS,1325 Fukuyama Street
361195,PATRICIA,JOHNSON,1325 Fukuyama Street


_Декартово произведение_ представляет *все* возможные сочетания записей из двух таблиц (599 клиентов \* 603 адреса = 361197 сочетаний). Этот тип соединения известен как _перекрестное соединение_ (`cross join`) и используется крайне редко.

### Внутренние соединения

In [7]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c JOIN address a
  ON c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,address
0,MARY,SMITH,1913 Hanoi Way
1,PATRICIA,JOHNSON,1121 Loja Avenue
2,LINDA,WILLIAMS,692 Joliet Street
3,BARBARA,JONES,1566 Inegl Manor
4,ELIZABETH,BROWN,53 Idfu Parkway
...,...,...,...
594,TERRENCE,GUNDERSON,844 Bucuresti Place
595,ENRIQUE,FORSYTHE,1101 Bucuresti Boulevard
596,FREDDIE,DUGGAN,1103 Quilmes Boulevard
597,WADE,DELVALLE,1331 Usak Boulevard


In [ ]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c INNER JOIN address a
  ON c.address_id = a.address_id;

Если имена столбцов, используемых для соединения двух таблиц, идентичны, вместо подпредложения `ON` можно использовать подпредложение `USING`

In [ ]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c INNER JOIN address a
  USING (address_id);

### Синтаксис соединения ANSI

In [10]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c, address a
WHERE c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,address
0,MARY,SMITH,1913 Hanoi Way
1,PATRICIA,JOHNSON,1121 Loja Avenue
2,LINDA,WILLIAMS,692 Joliet Street
3,BARBARA,JONES,1566 Inegl Manor
4,ELIZABETH,BROWN,53 Idfu Parkway
...,...,...,...
594,TERRENCE,GUNDERSON,844 Bucuresti Place
595,ENRIQUE,FORSYTHE,1101 Bucuresti Boulevard
596,FREDDIE,DUGGAN,1103 Quilmes Boulevard
597,WADE,DELVALLE,1331 Usak Boulevard


Преимущество синтаксиса соединения `SQL92` проще увидеть для сложных запросов, которые включают как условия соединения, так и условия фильтрации.

Старый синтаксис соединения ANSI:

In [12]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c, address a
WHERE c.address_id = a.address_id
  AND a.postal_code = 52137;

2 rows affected.

,first_name,last_name,address
0,JAMES,GANNON,1635 Kuwana Boulevard
1,FREDDIE,DUGGAN,1103 Quilmes Boulevard


Запрос с использованием синтаксиса соединения `SQL92`:

In [13]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c INNER JOIN address a
  ON c.address_id = a.address_id
WHERE a.postal_code = 52137;

2 rows affected.

,first_name,last_name,address
0,JAMES,GANNON,1635 Kuwana Boulevard
1,FREDDIE,DUGGAN,1103 Quilmes Boulevard


## Соединение трех и более таблиц

In [14]:
%%sql
desc address;

9 rows affected.

,Field,Type,Null,Key,Default,Extra
0,address_id,smallint unsigned,NO,PRI,NaN,auto_increment
1,address,varchar(50),NO,,NaN,
2,address2,varchar(50),YES,,NaN,
3,district,varchar(20),NO,,NaN,
4,city_id,smallint unsigned,NO,MUL,NaN,
5,postal_code,varchar(10),YES,,NaN,
6,phone,varchar(20),NO,,NaN,
7,location,geometry,NO,MUL,NaN,
8,last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [15]:
%%sql
desc city;

4 rows affected.

,Field,Type,Null,Key,Default,Extra
0,city_id,smallint unsigned,NO,PRI,NaN,auto_increment
1,city,varchar(50),NO,,NaN,
2,country_id,smallint unsigned,NO,MUL,NaN,
3,last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [21]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM customer c
  INNER JOIN address a
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = ct.city_id
-- ORDER BY ct.city;

599 rows affected.

,first_name,last_name,city
0,VIVIAN,RUIZ,´s-Hertogenbosch
1,JULIE,SANCHEZ,A Coruña (La Coruña)
2,PEGGY,MYERS,Abha
3,TOM,MILNER,Abu Dhabi
4,GLEN,TALBERT,Acuña
...,...,...,...
594,CONSTANCE,REID,Zaria
595,JACK,FOUST,Zeleznogorsk
596,BYRON,BOX,Zhezqazghan
597,GUY,BROWNLEE,Zhoushan


:::{tip} SQL _**не**_ является процедурным языком
:class: dropdown
:open: true

Вы описываете _**что**_ хотите получить и какие объекты базы данных должны быть вовлечены в запрос. Но как лучше выполнить ваш запрос – определяет сервер базы данных.

Поэтому приведенные ниже варианты запроса возвращают те же результаты:

In [ ]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM city ct
  INNER JOIN address a
  ON a.city_id = ct.city_id
  INNER JOIN customer c
  ON c.address_id = a.address_id;

In [ ]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM address a
  INNER JOIN city ct
  ON a.city_id = ct.city_id
  INNER JOIN customer c
  ON c.address_id = a.address_id;

Чтобы соединять таблицы в определенном порядке, можно разместить их в желаемом порядке и указать ключевое слово `straight_join`:

In [ ]:
%%sql
SELECT STRAIGHT_JOIN c.first_name, c.last_name, ct.city
FROM city ct
  INNER JOIN address a
  ON a.city_id = ct.city_id
  INNER JOIN customer c
  ON c.address_id = a.address_id;

### Использование подзапросов в качестве таблиц

In [27]:
%%sql
SELECT c.first_name, c.last_name, addr.address, addr.city
FROM customer c
  INNER JOIN
    (SELECT a.address_id, a.address, ct.city
    FROM address a
      INNER JOIN city ct
      ON a.city_id = ct.city_id
    WHERE a.district = 'California'
    ) addr
  ON c.address_id = addr.address_id;

9 rows affected.

,first_name,last_name,address,city
0,PATRICIA,JOHNSON,1121 Loja Avenue,San Bernardino
1,BETTY,WHITE,770 Bydgoszcz Avenue,Citrus Heights
2,ALICE,STEWART,1135 Izumisano Parkway,Fontana
3,ROSA,REYNOLDS,793 Cam Ranh Avenue,Lancaster
4,RENEE,LANE,533 al-Ayn Boulevard,Compton
5,KRISTIN,JOHNSTON,226 Brest Manor,Sunnyvale
6,CASSANDRA,WALTERS,920 Kumbakonam Loop,Salinas
7,JACOB,LANCE,1866 al-Qatif Avenue,El Monte
8,RENE,MCALISTER,1895 Zhezqazghan Drive,Garden Grove


:::{note} Примечание

Хотя этот запрос можно было бы написать без использования подзапроса, просто соединив три таблицы:
```sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
  INNER JOIN address a
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = ct.city_id
WHERE a.district = 'California';
```
иногда применение подзапроса может быть выгодным с точки зрения производительности и/или удобочитаемости.

Один из способов визуализировать происходящее – выполнить подзапрос сам по себе и посмотреть на результат:

In [30]:
%%sql
SELECT a.address_id, a.address, ct.city
FROM address a
  INNER JOIN city ct
  ON a.city_id = ct.city_id
WHERE a.district = 'California';

9 rows affected.

,address_id,address,city
0,6,1121 Loja Avenue,San Bernardino
1,18,770 Bydgoszcz Avenue,Citrus Heights
2,55,1135 Izumisano Parkway,Fontana
3,116,793 Cam Ranh Avenue,Lancaster
4,186,533 al-Ayn Boulevard,Compton
5,218,226 Brest Manor,Sunnyvale
6,274,920 Kumbakonam Loop,Salinas
7,425,1866 al-Qatif Avenue,El Monte
8,599,1895 Zhezqazghan Drive,Garden Grove


### Использование одной таблицы дважды

Найдем все фильмы, в которых снимались _Cate McQueen_ или _Cuba Birch_.

::::{seealso} Посмотрим в исходные таблицы
:class: dropdown
:open: true

```bash
SELECT film_id, title, release_year
FROM film;

| film_id | title            | release_year |
| ------- | ---------------- | ------------ |
| 1       | ACADEMY DINOSAUR | 2006         |
| 2       | ACE GOLDFINGER   | 2006         |
| 3       | ADAPTATION HOLES | 2006         |
| 4       | AFFAIR PREJUDICE | 2006         |
| 5       | AFRICAN EGG      | 2006         |
| ...     | ...              | ...          |
1000 rows × 3 columns
```

```bash
SELECT * FROM actor;

| actor_id | first_name | last_name    | last_update         |
| -------- | ---------- | ------------ | ------------------- |
| 1        | PENELOPE   | GUINESS      | 2006-02-15 04:34:33 |
| 2        | NICK       | WAHLBERG     | 2006-02-15 04:34:33 |
| 3        | ED         | CHASE        | 2006-02-15 04:34:33 |
| 4        | JENNIFER   | DAVIS        | 2006-02-15 04:34:33 |
| 5        | JOHNNY     | LOLLOBRIGIDA | 2006-02-15 04:34:33 |
| ...      | ...        | ...          | ...                 |
200 rows × 4 columns
```

```bash
SELECT * FROM film_actor;

| actor_id | film_id | last_update         |
| -------- | ------- | ------------------- |
| 1        | 1       | 2006-02-15 05:05:03 |
| 1        | 23      | 2006-02-15 05:05:03 |
| 1        | 25      | 2006-02-15 05:05:03 |
| 1        | 106     | 2006-02-15 05:05:03 |
| 1        | 140     | 2006-02-15 05:05:03 |
| ...      | ...     | ...                 |
5462 rows × 3 columns
```

:::{note} Pandas vs Polars
:class: dropdown
:open: true

Поскольку в **Pandas** стандартный лимит на максимальное количество отображаемых строк `max_rows` равен **60 строкам**, то в следующем запросе он не станет применять усечение через многоточие и вернет 54 строки запроса целиком.

Реализовать усечение можно двумя способами.
1. Воспользоваться **Polars** в котором по умолчанию установлены более строгие лимиты на количество строк `tbl_rows` равный **10 строкам**.
2. Или продолжить использовать **Pandas** предварительно уменьшив порог `max_rows`, например, до 10 строк:
```python
pd.set_option('display.max_rows', 10)
```
:::

In [12]:
# Способ 1: используем Polars
import polars as pl
print(pl.__version__)

1.43.2


In [19]:
%config SqlMagic.autopolars = True

Disabled 'autopandas' since 'autopolars' was enabled.

In [14]:
%%sql
SELECT f.title
FROM film f
  INNER JOIN film_actor fa
  ON f.film_id = fa.film_id
  INNER JOIN actor a
  ON fa.actor_id = a.actor_id
WHERE ((a.first_name = 'CATE' AND a.last_name = 'MCQUEEN')
    OR (a.first_name = 'CUBA' AND a.last_name = 'BIRCH'));

54 rows affected.

title
str
"""ATLANTIS CAUSE"""
"""BLOOD ARGONAUTS"""
"""COMMANDMENTS EXPRESS"""
"""DYNAMITE TARZAN"""
"""EDGE KISSING"""
…
"""TOWERS HURRICANE"""
"""TROJAN TOMORROW"""
"""VIRGIN DAISY"""


In [4]:
# Способ 2: в Pandas уменьшаем порог срабатывания усечения строк
pd.set_option('display.max_rows', 10)

In [20]:
%config SqlMagic.autopandas = True

Disabled 'autopolars' since 'autopandas' was enabled.

In [7]:
%%sql
SELECT f.title
FROM film f
  INNER JOIN film_actor fa
  ON f.film_id = fa.film_id
  INNER JOIN actor a
  ON fa.actor_id = a.actor_id
WHERE ((a.first_name = 'CATE' AND a.last_name = 'MCQUEEN')
    OR (a.first_name = 'CUBA' AND a.last_name = 'BIRCH'));

54 rows affected.

,title
0,ATLANTIS CAUSE
1,BLOOD ARGONAUTS
2,COMMANDMENTS EXPRESS
3,DYNAMITE TARZAN
4,EDGE KISSING
...,...
49,TOWERS HURRICANE
50,TROJAN TOMORROW
51,VIRGIN DAISY
52,VOLCANO TEXAS


Найдем только те фильмы, в которых появляются оба актера  \
_(одни и те же таблицы используются несколько раз)._

In [10]:
%config SqlMagic.autopandas = False

In [16]:
%%sql
SELECT f.title
FROM film f
  INNER JOIN film_actor fa1
  ON f.film_id = fa1.film_id
  INNER JOIN actor a1
  ON fa1.actor_id = a1.actor_id
  INNER JOIN film_actor fa2
  ON f.film_id = fa2.film_id
  INNER JOIN actor a2
  ON fa2.actor_id = a2.actor_id
WHERE (a1.first_name = 'CATE' AND a1.last_name = 'MCQUEEN')
    AND (a2.first_name = 'CUBA' AND a2.last_name = 'BIRCH');

2 rows affected.

title
BLOOD ARGONAUTS
TOWERS HURRICANE


### Самосоединение

```bash
SELECT f.title, f_prnt.title prequel
FROM film f
  INNER JOIN film f_prnt
  ON f_prnt.film_id = f.prequel_film_id
WHERE f.prequel_film_id IS NOT NULL;


| title           | prequel      |
| --------------- | ------------ |
| FIDDLER LOST II | FIDDLER LOST |
```

---

## Упражнения

### Упражнение 5.1
Заполните пропущенные места (обозначенные как <\#>) в следующем запросе так, чтобы получить показанные результаты.

```sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
  INNER JOIN address <1>
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = <2>
WHERE a.district = 'California';
```
```text
| first_name | last_name | address                | city           |
| ---------- | --------- | ---------------------- | -------------- |
| PATRICIA   | JOHNSON   | 1121 Loja Avenue       | San Bernardino |
| BETTY      | WHITE     | 770 Bydgoszcz Avenue   | Citrus Heights |
| ALICE      | STEWART   | 1135 Izumisano Parkway | Fontana        |
| ROSA       | REYNOLDS  | 793 Cam Ranh Avenue    | Lancaster      |
| RENEE      | LANE      | 533 al-Ayn Boulevard   | Compton        |
| KRISTIN    | JOHNSTON  | 226 Brest Manor        | Sunnyvale      |
| CASSANDRA  | WALTERS   | 920 Kumbakonam Loop    | Salinas        |
| JACOB      | LANCE     | 1866 al-Qatif Avenue   | El Monte       |
| RENE       | MCALISTER | 1895 Zhezqazghan Drive | Garden Grove   |
```

In [19]:
%%sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
  INNER JOIN address a
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = ct.city_id
WHERE a.district = 'California';

9 rows affected.

first_name,last_name,address,city
PATRICIA,JOHNSON,1121 Loja Avenue,San Bernardino
BETTY,WHITE,770 Bydgoszcz Avenue,Citrus Heights
ALICE,STEWART,1135 Izumisano Parkway,Fontana
ROSA,REYNOLDS,793 Cam Ranh Avenue,Lancaster
RENEE,LANE,533 al-Ayn Boulevard,Compton
KRISTIN,JOHNSTON,226 Brest Manor,Sunnyvale
CASSANDRA,WALTERS,920 Kumbakonam Loop,Salinas
JACOB,LANCE,1866 al-Qatif Avenue,El Monte
RENE,MCALISTER,1895 Zhezqazghan Drive,Garden Grove


### Упражнение 5.2

Напишите запрос, который выводил бы названия всех фильмов, в которых играл актер с именем JOHN.

In [20]:
# Решение

### Упражнение 5.3

Создайте запрос, который возвращает все адреса в одном и том же городе. Вам нужно будет соединить таблицу адресов с самой собой, и каждая строка должна включать два разных адреса.

In [21]:
# Решение


---